# Análise Crítica do Kimi

**Objetivo**: Analisar relatório HTML do Gemini usando Kimi AI (via NVIDIA NIM) e fornecer uma análise crítica considerando também os dados CSV do portfólio.

**Fluxo**:
1. Lê relatório HTML (`/results/report.html`)
2. Lê dados CSV do portfólio (`/results/tikers_grouped.csv`)
3. Envia para Kimi com prompt de análise crítica
4. Exibe resultado e salva em arquivo
5. (Opcional) Envia por e-mail

**Requisitos**:
- `.env` com `KIMI_API_KEY` e `app_key`
- Arquivos HTML e CSV gerados previamente

|---|---|---|
| Modelo | Descrição | Use Case |
|---|---|---|
| moonshotai/kimi-k2.5	(atual) | Multimodal, Mixture-of-Experts com 1T params	| Análise geral, textos longos |
| moonshotai/kimi-k2-thinking	| Modelo com raciocínio explícito|	Problemas complexos, lógica |
| moonshotai/kimi-k2-instruct-0905|	Long-context (256k tokens)|	Documentos extensos, análise de código |
| moonshotai/kimi-k2-instruct|	Otimizado para coding|	Geração de código, debugging |

In [1]:
"""
Análise Crítica do Kimi - Análise de Relatório Gemini com Kimi AI

Este script analisa um relatório HTML gerado pelo Gemini usando a API Kimi (Moonshot AI)
via NVIDIA NIM. Ele também considera os dados CSV do portfólio para uma análise crítica.

Requisitos:
- Arquivo .env com KIMI_API_KEY
- Arquivo HTML do relatório Gemini em: /home/ponche/projects/ibox/reports/report.html
- Arquivo CSV com dados do portfólio em: /home/ponche/projects/ibox/results/tikers_grouped.csv
"""

import os
import re
import pandas as pd
from dotenv import load_dotenv

# Carrega variáveis de ambiente
load_dotenv()

# Constantes da API
KIMI_API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
KIMI_MODEL = "moonshotai/kimi-k2-instruct-0905"

# Caminhos dos arquivos
REPORT_PATH = "/home/ponche/projects/ibox/reports/report.html"
CSV_PATH = "/home/ponche/projects/ibox/results/tikers_grouped.csv"


def get_api_key():
    """Obtém e valida a API Key do Kimi."""
    api_key = os.getenv('KIMI_API_KEY')
    if not api_key:
        raise ValueError("KIMI_API_KEY não encontrada no .env!")
    print(f"✓ API Key carregada: {api_key[:10]}... ({len(api_key)} caracteres)")
    return api_key


def clean_html_content(html_content):
    """
    Remove tags não essenciais do HTML para reduzir payload.
    Preserva: estrutura, títulos, parágrafos, tabelas, listas.
    Remove: CSS, scripts, atributos de estilo, comentários.
    """
    if not html_content:
        return html_content
    
    # Remove tags <style>...</style>
    html_content = re.sub(r'<style[^>]*>.*?</style>', '', html_content, flags=re.DOTALL | re.IGNORECASE)
    
    # Remove tags <script>...</script>
    html_content = re.sub(r'<script[^>]*>.*?</script>', '', html_content, flags=re.DOTALL | re.IGNORECASE)
    
    # Remove comentários HTML <!-- ... -->
    html_content = re.sub(r'<!--.*?-->', '', html_content, flags=re.DOTALL)
    
    # Remove atributos de classe
    html_content = re.sub(r'\s+class="[^"]*"', '', html_content)
    html_content = re.sub(r"\s+class='[^']*'", '', html_content)
    
    # Remove atributos de style inline
    html_content = re.sub(r'\s+style="[^"]*"', '', html_content)
    html_content = re.sub(r"\s+style='[^']*'", '', html_content)
    
    # Remove atributos de id (geralmente usados para CSS)
    html_content = re.sub(r'\s+id="[^"]*"', '', html_content)
    html_content = re.sub(r"\s+id='[^']*'", '', html_content)
    
    # Remove atributos de data-*
    html_content = re.sub(r'\s+data-[a-zA-Z-]+="[^"]*"', '', html_content)
    html_content = re.sub(r"\s+data-[a-zA-Z-]+='[^']*'", '', html_content)
    
    # Remove múltiplos espaços em branco
    html_content = re.sub(r'\s+', ' ', html_content)
    
    # Remove espaços entre tags
    html_content = re.sub(r'>\s+<', '><', html_content)
    
    print(f"✓ HTML limpo: de {len(html_content)} para ~{len(html_content)} caracteres")
    return html_content.strip()


def read_html_report(path, max_chars=8000):
    """
    Lê o relatório HTML, limpa tags não essenciais e trunca se necessário.
    
    Args:
        path: Caminho do arquivo HTML
        max_chars: Máximo de caracteres a enviar para API
    
    Returns:
        String com conteúdo HTML limpo (possivelmente truncado)
    """
    try:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        print(f"✓ HTML original: {len(content)} caracteres")
        
        # Limpa o HTML
        content = clean_html_content(content)
        
        # Trunca se ainda for muito grande
        if len(content) > max_chars:
            content = content[:max_chars] + "\n...[truncado para limite de API]"
            print(f"⚠ Truncado para {max_chars} caracteres")
        
        return content
    except Exception as e:
        print(f"✗ Erro ao ler HTML: {e}")
        return None


def read_portfolio_data(path, top_n=10):
    """
    Lê o CSV do portfólio e retorna os principais ativos.
    
    Args:
        path: Caminho do arquivo CSV
        top_n: Número de ativos principais a retornar
    
    Returns:
        String CSV com os top N ativos
    """
    try:
        df = pd.read_csv(path)
        df_top = df.nlargest(top_n, 'PERCENTUAL')
        # Seleciona apenas colunas essenciais para reduzir payload
        essential_cols = ['TIKER', 'SETOR', 'PERCENTUAL', 'COTACAO', 'PL', 'P_V', 'ROE', 'DIV_YELD']
        available_cols = [col for col in essential_cols if col in df_top.columns]
        df_top = df_top[available_cols]
        
        csv_content = df_top.to_csv(index=False)
        print(f"✓ CSV carregado: top {top_n} ativos, {len(available_cols)} colunas ({len(csv_content)} caracteres)")
        return csv_content
    except Exception as e:
        print(f"✗ Erro ao ler CSV: {e}")
        return None


def call_kimi(prompt, max_tokens=4000, temperature=0.3, timeout=60):
    """
    Chama a API Kimi via NVIDIA NIM.
    
    Args:
        prompt: Texto do prompt
        max_tokens: Máximo de tokens na resposta
        temperature: Temperatura para geração (0-1)
        timeout: Timeout em segundos
    
    Returns:
        String com resposta da API ou None em caso de erro
    """
    import requests
    
    api_key = get_api_key()
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": KIMI_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": 1.0,
        "stream": False
    }
    
    print(f"  → Enviando requisição para Kimi...")
    print(f"  → Payload total: {len(prompt)} caracteres | Max tokens: {max_tokens} | Timeout: {timeout}s")
    
    try:
        response = requests.post(KIMI_API_URL, headers=headers, json=payload, timeout=timeout)
        response.raise_for_status()
        
        result = response.json()
        
        if 'error' in result:
            print(f"  ✗ Erro da API: {result['error']}")
            return None
        
        if not result.get('choices'):
            print(f"  ✗ Sem resposta da API")
            return None
        
        content = result['choices'][0].get('message', {}).get('content')
        finish_reason = result['choices'][0].get('finish_reason', 'unknown')
        
        print(f"  ✓ Resposta recebida: {len(content)} caracteres (finish_reason: {finish_reason})")
        return content
        
    except requests.exceptions.Timeout:
        print(f"  ✗ Timeout após {timeout} segundos")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Erro de conexão: {e}")
        return None
    except Exception as e:
        print(f"  ✗ Erro inesperado: {e}")
        return None


def create_analysis_prompt(html_content, csv_content):
    """
    Cria o prompt para análise crítica do relatório.
    
    Args:
        html_content: Conteúdo do relatório HTML (já limpo)
        csv_content: Dados CSV do portfólio
    
    Returns:
        String com o prompt completo
    """
    return f"""Você é um analista de investimentos experiente. Analise este relatório de portfólio e os dados CSV fornecidos.

TAREFA:
1. Avalie se recomendações fazem sentido pelos indicadores
2. Sugira melhorias e elogie o que vc concorda
3. Leve em consideracao a guerra no golfo persico e os impactos no petroleo, juros e fertilizantes. Lebrando que o Brasil produz etanol.
4. Execessoes:
    - vulc3 pays only 1% of yeal as statutory dividend change
    - brap4 is like vale3 it is company that only have vale3 stoks

Seja conciso.

---
RELATÓRIO HTML:
{html_content}

---
DADOS PORTFÓLIO (CSV):
{csv_content}"""


def analyze_report(report_path, csv_path):
    """
    Função principal que orquestra a análise do relatório.
    
    Args:
        report_path: Caminho do arquivo HTML
        csv_path: Caminho do arquivo CSV
    
    Returns:
        String com análise do Kimi ou mensagem de erro
    """
    print("=" * 60)
    print("ANÁLISE CRÍTICA DO KIMI")
    print("=" * 60)
    
    # 1. Lê os arquivos
    html_content = read_html_report(report_path)
    csv_content = read_portfolio_data(csv_path)
    
    if not html_content or not csv_content:
        return "Erro: Não foi possível ler os arquivos necessários."
    
    # 2. Cria o prompt
    prompt = create_analysis_prompt(html_content, csv_content)
    
    # 3. Chama a API
    print("\n" + "-" * 60)
    print("Enviando para análise do Kimi...")
    print("-" * 60)
    
    analysis = call_kimi(prompt, max_tokens=4000, temperature=0.3, timeout=300)
    
    return analysis


# ============================================
# EXECUÇÃO PRINCIPAL
# ============================================

resultado = analyze_report(REPORT_PATH, CSV_PATH)

if resultado:
    print("\n" + "=" * 60)
    print("ANÁLISE CRÍTICA DO KIMI:")
    print("=" * 60)
    print(resultado)
else:
    print("\n✗ Falha na análise - verifique logs acima")

ANÁLISE CRÍTICA DO KIMI
✓ HTML original: 16357 caracteres
✓ HTML limpo: de 8803 para ~8803 caracteres
⚠ Truncado para 8000 caracteres
✓ CSV carregado: top 10 ativos, 8 colunas (851 caracteres)

------------------------------------------------------------
Enviando para análise do Kimi...
------------------------------------------------------------
✓ API Key carregada: nvapi-K0HE... (70 caracteres)
  → Enviando requisição para Kimi...
  → Payload total: 9456 caracteres | Max tokens: 4000 | Timeout: 300s
  ✓ Resposta recebida: 1395 caracteres (finish_reason: stop)

ANÁLISE CRÍTICA DO KIMI:
**Parecer – 5 min, 4 pontos-chave**

1. **Recomendações do relatório estão corretas.**  
   - VULC3 paga apenas 1% de dividendos estatutários → yield de 27,9% é “falso”.  
   - AZZA3, JHSF3, ROMI3: setores cíclicos com ROE < 15% e sensíveis a juros altos → vender é o caminho.  
   - PETR4, BBAS3, TAEE4, ABCB4, BBSE3: cash-flow blindado, yield > 6% e beneficiam-se da Selic alta → manter/acumular faz sent

# Exibir e Salvar Análise Crítica do Kimi

Esta célula exibe a análise crítica e salva em `/reports/report_kimi.html`

In [2]:
"""
Exibe e salva a Análise Crítica do Kimi em HTML formatado.
Pega a análise em texto e chama o Kimi para reformatar em HTML amigável.
"""

import re
from IPython.display import display, HTML

# Configuração
OUTPUT_FILE = '/home/ponche/projects/ibox/reports/KIMI_FEEDBACK_FROM_GEMINI.html'


def format_analysis_as_html(analysis_text):
    """
    Chama Kimi para reformatar a análise em HTML amigável.
    """
    import requests
    
    api_key = os.getenv('KIMI_API_KEY')
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""Reformate esta análise de portfólio em HTML limpo e amigável para humanos.

REQUISITOS:
- Use HTML semântico com tags modernas
- Estilo inline simples (fonts, cores, padding)
- Títulos em destaque, parágrafos bem espaçados
- Use cards ou seções claras para cada ponto
- Destaque positivos em verde, negativos em vermelho/laranja
- Adicione emojis apropriados
- Layout responsivo e legível
- NÃO use CSS externo ou frameworks

CONTEÚDO PARA FORMATAR:
{analysis_text}

Gere apenas o HTML, sem explicações."""
    
    payload = {
        "model": "moonshotai/kimi-k2.5",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 6000,
        "temperature": 0.3,
        "top_p": 1.0,
        "stream": False
    }
    
    print("  → Chamando Kimi para formatar HTML...")
    
    try:
        response = requests.post(
            "https://integrate.api.nvidia.com/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        response.raise_for_status()
        
        result = response.json()
        
        if 'error' in result:
            print(f"  ✗ Erro da API: {result['error']}")
            return None
        
        if not result.get('choices'):
            return None
        
        content = result['choices'][0].get('message', {}).get('content')
        print(f"  ✓ HTML formatado: {len(content)} caracteres")
        return content
        
    except Exception as e:
        print(f"  ✗ Erro ao formatar: {e}")
        return None


def clean_html_output(html_content):
    """Remove markdown code blocks se existirem."""
    if not html_content or not isinstance(html_content, str):
        return None
    
    cleaned = re.sub(r'^```html\s*', '', html_content, flags=re.IGNORECASE)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    return cleaned.strip()


def save_report(html_content, filepath):
    """Salva o relatório em arquivo HTML."""
    try:
        with open(filepath, "w", encoding='utf-8') as f:
            f.write(html_content)
        print(f"✓ Análise salva em: {filepath}")
        return True
    except Exception as e:
        print(f"✗ Erro ao salvar: {e}")
        return False


# --- EXECUÇÃO ---

# 1. Verifica se temos a análise
if 'html_report_kimi' not in globals() or not html_report_kimi:
    print("✗ Erro: Execute a célula de análise primeiro!")
else:
    analysis_text = html_report_kimi
    
    print("=" * 60)
    print("FORMATANDO ANÁLISE EM HTML")
    print("=" * 60)
    
    # 2. Chama Kimi para formatar
    html_formatted = format_analysis_as_html(analysis_text)
    
    if html_formatted:
        # 3. Limpa o HTML
        html_clean = clean_html_output(html_formatted)
        
        # 4. Salva
        save_report(html_clean, OUTPUT_FILE)
        
        # 5. Exibe
        print("\n--- Preview ---")
        display(HTML(html_clean))
    else:
        print("\n✗ Falha ao formatar - exibindo texto original")
        print(analysis_text)

✗ Erro: Execute a célula de análise primeiro!


# Enviar Análise Crítica do Kimi por E-mail

Envia a análise crítica via Gmail SMTP.

In [3]:
"""
Envia o relatório por e-mail usando Gmail SMTP.
Requer: app_key no arquivo .env (senha de app do Gmail)
"""

import smtplib
import os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


# --- CONFIGURAÇÕES ---
EMAIL_REMETENTE = "marcos.ponche@gmail.com"
EMAIL_DESTINO = "marcos.ponche@gmail.com"
ASSUNTO = "📈 Análise Crítica do Kimi"


def get_app_password():
    """Obtém a senha de app do Gmail do .env."""
    password = os.getenv('app_key')
    if not password:
        raise ValueError("app_key não encontrada no .env!")
    return password


def create_email(html_content, remetente, destino, assunto):
    """Cria a estrutura do e-mail com conteúdo HTML."""
    msg = MIMEMultipart('alternative')
    msg['Subject'] = assunto
    msg['From'] = remetente
    msg['To'] = destino
    
    parte_html = MIMEText(html_content, 'html')
    msg.attach(parte_html)
    
    return msg


def send_email_smtp(msg, remetente, password):
    """Conecta ao Gmail e envia o e-mail."""
    try:
        print("Conectando ao servidor SMTP do Gmail...")
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(remetente, password)
        
        print("Enviando e-mail...")
        server.sendmail(remetente, msg['To'], msg.as_string())
        server.quit()
        
        print(f"✅ E-mail enviado com sucesso para {msg['To']}")
        return True
        
    except smtplib.SMTPAuthenticationError:
        print("❌ Erro de autenticação - verifique a app_key no .env")
    except smtplib.SMTPException as e:
        print(f"❌ Erro SMTP: {e}")
    except Exception as e:
        print(f"❌ Erro inesperado: {e}")
    
    return False


# --- EXECUÇÃO ---

# Verifica se temos o relatório
if 'html_report' not in globals() or not html_report:
    print("✗ Erro: Execute a célula de exibição primeiro!")
else:
    try:
        # Obtém credenciais
        app_password = get_app_password()
        
        # Cria e envia o e-mail
        email_msg = create_email(html_report, EMAIL_REMETENTE, EMAIL_DESTINO, ASSUNTO)
        send_email_smtp(email_msg, EMAIL_REMETENTE, app_password)
        
    except Exception as e:
        print(f"✗ Falha no envio: {e}")

✗ Erro: Execute a célula de exibição primeiro!
